# LangChain Fundamentals
### Setting up models, sending messages, and understanding LangChain's core primitives

---
**Topics Covered:**
- Installing dependencies
- Loading API keys from `.env`
- Initializing LLMs via `init_chat_model`
- Message types: `SystemMessage`, `HumanMessage`, `AIMessage`
- `invoke()` with different prompts
- Inspecting `AIMessage` response metadata


## 1. Install Required Packages

In [1]:
# Run once to install dependencies
# !pip install langchain langchain-groq langchain-google-genai python-dotenv

## 2. Load Environment Variables

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY    = os.getenv("GROQ_API_KEY")
GOOGLE_API_KEY  = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY")

# Quick sanity check — prints True if key was loaded
print("Groq key loaded  :", bool(GROQ_API_KEY))
print("Google key loaded:", bool(GOOGLE_API_KEY))

Groq key loaded  : True
Google key loaded: True


## 3. Initialize a Chat Model with `init_chat_model`

`init_chat_model` is a provider-agnostic factory that returns a `BaseChatModel`.  
Swap the model name/provider string without touching the rest of your code.

In [3]:
from langchain.chat_models import init_chat_model

# Using an open-source model served on Groq's inference API
llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq",
    temperature=0.7,
)

print(type(llm))

<class 'langchain_groq.chat_models.ChatGroq'>


## 4. Understanding LangChain Message Types

| Class | Role | When to use |
|-------|------|-------------|
| `SystemMessage` | system | Sets the assistant's persona / constraints |
| `HumanMessage` | user | The user's question or instruction |
| `AIMessage` | assistant | The model's previous reply (in multi-turn) |

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Build a conversation manually
conversation = [
    SystemMessage(content="You are a concise Python tutor. Keep answers under 80 words."),
    HumanMessage(content="What is a Python decorator and why would I use one?"),
]

response = llm.invoke(conversation)
print(response.content)

A Python decorator is a function that modifies another function. Use it to add functionality without altering the original code, such as logging, authentication, or caching. It wraps the original function, allowing for code reuse and flexibility.


## 5. Inspecting the `AIMessage` Object

The returned object carries more than just `.content` — token counts, finish reason, model name, and latency are all accessible.

In [5]:
print("Type          :", type(response))
print("Finish reason :", response.response_metadata.get("finish_reason"))
print("Model used    :", response.response_metadata.get("model_name"))
print("Input tokens  :", response.usage_metadata.get("input_tokens"))
print("Output tokens :", response.usage_metadata.get("output_tokens"))

Type          : <class 'langchain_core.messages.ai.AIMessage'>
Finish reason : stop
Model used    : llama-3.3-70b-versatile
Input tokens  : 61
Output tokens : 45


## 6. Multi-Turn Conversation

Append the model's last reply as an `AIMessage` to maintain conversational context across turns.

In [6]:
# Turn 1
history = [
    SystemMessage(content="You are a concise Python tutor."),
    HumanMessage(content="Explain list comprehension in one sentence."),
]
turn1 = llm.invoke(history)
print("Assistant (turn 1):", turn1.content)

# Turn 2 — append previous reply and ask a follow-up
history.append(turn1)   # AIMessage
history.append(HumanMessage(content="Show me a quick example with squares of even numbers."))

turn2 = llm.invoke(history)
print("\nAssistant (turn 2):", turn2.content)

Assistant (turn 1): List comprehension is a concise way to create lists in Python by performing an operation on each item in an iterable and returning a new list with the results.



Assistant (turn 2): ```python
numbers = [1, 2, 3, 4, 5, 6]
even_squares = [x**2 for x in numbers if x % 2 == 0]
print(even_squares)  # Output: [4, 16, 36]
```


## 7. Swapping Models — Another Groq Model

The same `invoke()` interface works no matter which model you pick. Just change the model name string passed to `init_chat_model` — here we swap to the smaller, faster `llama-3.1-8b-instant`.

In [7]:
fast_llm = init_chat_model(
    "llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0.3,
)

answer = fast_llm.invoke("Name three lesser-known Python standard-library modules and what each does.")
print(answer.content)

Here are three lesser-known Python standard-library modules and what each does:

1. **`math` module's `hypot` function and `isqrt` function**: The `math` module is well-known, but some of its functions are not as widely used. The `hypot` function calculates the Euclidean norm (magnitude) of a vector given in Cartesian coordinates, which is useful in geometry and physics. The `isqrt` function calculates the integer square root of a number, which is useful in various mathematical and computational applications.

2. **`functools` module's `cmp_to_key` function**: The `functools` module is often used for higher-order functions, but the `cmp_to_key` function is not as well-known. This function converts an old-style comparison function (which takes two arguments and returns a value indicating their relationship) into a key function that can be used with the `sorted` function or other functions that require a key function.

3. **`itertools` module's `cycle` function**: The `itertools` module 

## 8. Plain String Shortcut

Passing a raw string to `invoke()` is the same as wrapping it in a single `HumanMessage` — handy for quick experiments.

In [8]:
quick = llm.invoke("In three bullet points, explain how garbage collection works in CPython.")
print(quick.content)

Here's an overview of how garbage collection works in CPython:
* **Reference Counting**: CPython uses a reference counting algorithm to manage memory. When an object is created, its reference count is set to 1. Every time a reference to the object is created (e.g., assigning it to a variable or passing it to a function), the reference count is incremented. When a reference to the object is removed (e.g., the variable goes out of scope or is reassigned), the reference count is decremented. If the reference count reaches 0, the object is immediately deallocated.
* **Cycle Detection**: While reference counting is efficient, it can't handle cyclic references (i.e., objects that reference each other). To address this, CPython uses a cycle detection algorithm that runs periodically. This algorithm identifies cycles of unreachable objects and breaks them, allowing the objects to be deallocated.
* **Generational Collection**: CPython's garbage collector also uses a generational approach, divid

---
### Summary

| Concept | Key class / method |
|---------|--------------------|
| Model factory | `init_chat_model(model, model_provider)` |
| Send messages | `llm.invoke([...messages...])` |
| Conversation context | Append `AIMessage` between turns |
| Token / metadata | `response.usage_metadata`, `response.response_metadata` |

**Next → `02_prompt_templates.ipynb`**